# Comparison

## Loading Dataset

In [1]:
from datasets import load_dataset
import sys
import os

sys.path.append(os.path.abspath(os.path.join('..')))

from config import DATASET_NAME

dataset = load_dataset(DATASET_NAME)
all_data = dataset["train"]

train_samples = [{"image": x["image_path"], "text": x["text"]} for x in all_data if x["split"] == "train"]
val_samples = [{"image": x["image_path"], "text": x["text"]} for x in all_data if x["split"] == "val"]
test_samples = [{"image": x["image_path"], "text": x["text"]} for x in all_data if x["split"] == "test"]

print("Successfully loaded dataset")
print(f"Train: {len(train_samples)} | Val: {len(val_samples)} | Test: {len(test_samples)}")

Successfully loaded dataset
Train: 235 | Val: 50 | Test: 51


## Loading Pretrained and Fine-tuned models

In [2]:
from transformers import TrOCRProcessor, VisionEncoderDecoderModel, GenerationConfig
from PIL import Image, ImageOps
from config import PT_MODEL_NAME, FT_MODEL_NAME, get_device
import torch

# Load pretrained model
pretrained_processor = TrOCRProcessor.from_pretrained(PT_MODEL_NAME)
pretrained_model = VisionEncoderDecoderModel.from_pretrained(PT_MODEL_NAME, torch_dtype=torch.float32)
pretrained_model.eval()

# Load fine-tuned model from Hub
finetuned_processor = TrOCRProcessor.from_pretrained(FT_MODEL_NAME)
finetuned_model = VisionEncoderDecoderModel.from_pretrained(FT_MODEL_NAME, torch_dtype=torch.float32)
finetuned_model.eval()

device = torch.device(get_device())

pretrained_model.to(device)
finetuned_model.to(device)

# Fix token IDs on pretrained model
pretrained_model.config.decoder_start_token_id = 1
pretrained_model.config.pad_token_id = 1
pretrained_model.config.eos_token_id = 2
pretrained_model.generation_config = GenerationConfig(
    decoder_start_token_id=1,
    eos_token_id=2,
    pad_token_id=1,
    max_new_tokens=64,
    forced_eos_token_id=None,
    forced_bos_token_id=None,
)
pretrained_model.decoder.config.decoder_start_token_id = 1
pretrained_model.decoder.config.eos_token_id = 2

print("Successfully loaded models")
print(f"device:", device)

Loading weights:   0%|          | 0/360 [00:00<?, ?it/s]

VisionEncoderDecoderModel LOAD REPORT from: microsoft/trocr-small-handwritten
Key                         | Status  | 
----------------------------+---------+-
encoder.pooler.dense.bias   | MISSING | 
encoder.pooler.dense.weight | MISSING | 

Notes:
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


Loading weights:   0%|          | 0/362 [00:00<?, ?it/s]

Successfully loaded models
device: mps


In [ ]:
def predict(model, processor, sample):
    image = sample["image"].convert("RGB")
    
    image = ImageOps.grayscale(image)
    image = ImageOps.autocontrast(image)
    image = image.convert("RGB")
    pixel_values = processor(images=image, return_tensors="pt").pixel_values.to(device).float()
    with torch.no_grad():
        generated_ids = model.generate(pixel_values)
    return processor.batch_decode(generated_ids, skip_special_tokens=True)[0]


def compare(sample):
    pretrained_out = predict(pretrained_model, pretrained_processor, sample)
    finetuned_out = predict(finetuned_model,  finetuned_processor,  sample)
    print(f"Ground truth: {sample['text']}")
    print(f"Pretrained:   {pretrained_out}")
    print(f"Fine-tuned:   {finetuned_out}")
    print()

print("Loaded comparison functions")

Loaded comparison functions


## Prediction Viz

In [4]:
from jiwer import cer, wer

pretrained_predictions = []
finetuned_predictions  = []
ground_truths          = []

print("decoder_start_token_id:", pretrained_model.config.decoder_start_token_id)
print("decoder config start:  ", pretrained_model.decoder.config.decoder_start_token_id)

for i, sample in enumerate(test_samples):
    pt_pred = predict(pretrained_model, pretrained_processor, sample)
    ft_pred = predict(finetuned_model,  finetuned_processor,  sample)
    
    pretrained_predictions.append(pt_pred)
    finetuned_predictions.append(ft_pred)
    ground_truths.append(sample["text"])
    
    print(f"{i+1}/51 done")

print("\nPretrained — CER:", f"{cer(ground_truths, pretrained_predictions):.2%}", "| WER:", f"{wer(ground_truths, pretrained_predictions):.2%}")
print("Fine-tuned  — CER:", f"{cer(ground_truths, finetuned_predictions):.2%}",  "| WER:", f"{wer(ground_truths, finetuned_predictions):.2%}")

decoder_start_token_id: 1
decoder config start:   1
1/51 done
2/51 done
3/51 done
4/51 done
5/51 done
6/51 done
7/51 done
8/51 done
9/51 done
10/51 done
11/51 done
12/51 done
13/51 done
14/51 done
15/51 done
16/51 done
17/51 done
18/51 done
19/51 done
20/51 done
21/51 done
22/51 done
23/51 done
24/51 done
25/51 done
26/51 done
27/51 done
28/51 done
29/51 done
30/51 done
31/51 done
32/51 done
33/51 done
34/51 done
35/51 done
36/51 done
37/51 done
38/51 done
39/51 done
40/51 done
41/51 done
42/51 done
43/51 done
44/51 done
45/51 done
46/51 done
47/51 done
48/51 done
49/51 done
50/51 done
51/51 done

Pretrained — CER: 381.35% | WER: 551.91%
Fine-tuned  — CER: 7.75% | WER: 21.86%


## Training Process Viz

In [ ]:
import json
import matplotlib.pyplot as plt
import os

# Load all history files
def load_history(filename):
    with open(f"results/metrics/{filename}", "r") as f:
        return json.load(f)

history_1 = load_history("lr=5e-5_augTrue_5ep.json")
history_2 = load_history("lr=1e-5_augTrue_5ep.json")
history_3 = load_history("lr=1e-4_augTrue_5ep.json")
history_4 = load_history("lr=5e-5_augFalse_5ep.json")
history_best = load_history("lr=1e-5_augTrue_15ep.json")

os.makedirs("results/figures", exist_ok=True)

# Plot 1 — Loss curve for best experiment
plt.figure(figsize=(10, 5))
plt.plot(range(1, 16), history_best["train_loss"], marker="o", color="steelblue", label="Training Loss")
plt.xlabel("Epoch")
plt.ylabel("Loss")
plt.title("Training Loss Over 15 Epochs (lr=1e-5, augment=True)")
plt.xticks(range(1, 16))
plt.legend()
plt.tight_layout()
plt.savefig("results/figures/loss_curve_best.png", dpi=150)
plt.show()

# Plot 2 — Val CER across all experiments
plt.figure(figsize=(10, 5))
plt.plot(range(1, 6), history_1["val_cer"], marker="o", label="lr=5e-5, augment=True")
plt.plot(range(1, 6), history_2["val_cer"], marker="o", label="lr=1e-5, augment=True")
plt.plot(range(1, 6), history_3["val_cer"], marker="o", label="lr=1e-4, augment=True")
plt.plot(range(1, 6), history_4["val_cer"], marker="o", label="lr=5e-5, augment=False")
plt.xlabel("Epoch")
plt.ylabel("Val CER")
plt.title("Validation CER Across All Experiments")
plt.legend()
plt.tight_layout()
plt.savefig("results/figures/val_cer_all_experiments.png", dpi=150)
plt.show()